# 🩺 Despliegue del modelo predictivo — Interfaz gráfica (Jupyter)

**Proyecto:** Predictor de Riesgo de Diabetes (Pima Indians Diabetes, NIDDK)  
**Modelo cargado:** `modelo_Xgboost.pkl` — `Pipeline(MinMaxScaler + XGBClassifier(n_estimators=50))`  
**Calidad en test (30%):** Accuracy **0.769** · Precisión 0.696 · Recall 0.600 · F1 0.644 · ROC-AUC 0.807  
**Umbral de decisión:** 0.5

> Este notebook es el **entregable de despliegue con interfaz gráfica**: controles interactivos (`ipywidgets`) para predicción individual, carga de CSV para predicción por lote y visualización de métricas. El despliegue web equivalente está en `app.py` (Streamlit).

In [ ]:
# --- 1. Configuración y carga del modelo ---
import pickle, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
BASE = Path(".")
FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "BMI",
            "DiabetesPedigreeFunction", "Age"]
LABELS = {"Pregnancies": "Embarazos", "Glucose": "Glucosa (mg/dL)",
          "BloodPressure": "Presión arterial (mm Hg)", "BMI": "IMC (kg/m²)",
          "DiabetesPedigreeFunction": "Antecedente familiar (DPF)", "Age": "Edad (años)"}
UMBRAL = 0.5

with open(BASE / "modelo_Xgboost.pkl", "rb") as f:
    modelo = pickle.load(f)
print(type(modelo).__name__, "|", modelo.steps)

tabla = pd.read_csv(BASE / "resultados_medidas.csv", index_col=0)
print(tabla.round(3).to_string())

## 🔮 2. Interfaz gráfica — predicción individual
Mueve los deslizadores y pulsa **Predecir riesgo**.

In [ ]:
# --- 2. Controles ---
sliders = {
    "Pregnancies": widgets.IntSlider(value=2, min=0, max=17, description="Embarazos"),
    "Glucose": widgets.IntSlider(value=120, min=40, max=250, description="Glucosa"),
    "BloodPressure": widgets.IntSlider(value=72, min=30, max=130, description="Presión"),
    "BMI": widgets.FloatSlider(value=28.5, min=15.0, max=60.0, step=0.1, description="IMC"),
    "DiabetesPedigreeFunction": widgets.FloatSlider(value=0.45, min=0.05, max=2.5, step=0.01, description="DPF"),
    "Age": widgets.IntSlider(value=32, min=18, max=85, description="Edad"),
}
boton = widgets.Button(description="🔍 Predecir riesgo", button_style="success")
salida = widgets.Output()
display(widgets.VBox(list(sliders.values()) + [boton, salida]))

In [ ]:
# --- 3. Lógica de predicción + gráfico ---
def predecir(datos: pd.DataFrame):
    proba = modelo.predict_proba(datos[FEATURES])[:, 1]
    return proba, (proba >= UMBRAL).astype(int)

def mostrar_resultado(fila: dict):
    df = pd.DataFrame([fila], columns=FEATURES)
    p = float(predecir(df)[0][0])
    clase = "DIABETES ⚠️" if p >= UMBRAL else "NO DIABETES ✅"
    color = "#B42318" if p >= UMBRAL else "#147A3E"
    display(HTML(f"<h3>Probabilidad de diabetes: <span style='color:{color}'>{p:.3f} ({p*100:.1f}%)</span> → <b>{clase}</b></h3>"))
    fig, ax = plt.subplots(figsize=(6, 1.6))
    ax.barh(["Riesgo"], [p], color=color)
    ax.set_xlim(0, 1); ax.axvline(UMBRAL, color="black", linestyle="--", label=f"Umbral {UMBRAL}")
    ax.set_xlabel("P(diabetes)"); ax.legend(); plt.tight_layout(); plt.show()
    return p

def al_pulsar(_):
    with salida:
        salida.clear_output()
        mostrar_resultado({k: w.value for k, w in sliders.items()})

boton.on_click(al_pulsar)

# Demostración automática (caso del informe: 2, 120, 78, 28.5, 0.45, 32 → P≈0.208)
demo = {"Pregnancies": 2, "Glucose": 120, "BloodPressure": 78,
        "BMI": 28.5, "DiabetesPedigreeFunction": 0.45, "Age": 32}
print("Caso demo del informe:")
mostrar_resultado(demo)

## 📦 4. Predicción por lote (CSV)
Se califican los 5 casos futuros del informe (`predicciones_futuras.csv`). Para tus propios datos, sube un CSV con las 6 columnas usando el control de abajo.

In [ ]:
# --- 4a. Lote de ejemplo incluido en el repo ---
lote = pd.read_csv(BASE / "predicciones_futuras.csv")
X_lote = lote[FEATURES]
pb, pr = predecir(X_lote)
lote_out = X_lote.copy()
lote_out["P(diabetes)"] = np.round(pb, 4)
lote_out["Predicción"] = np.where(pr == 1, "DIABETES", "NO DIABETES")
display(lote_out)
lote_out.to_csv(BASE / "predicciones_lote_notebook.csv", index=False)
print("Guardado: predicciones_lote_notebook.csv")

In [ ]:
# --- 4b. Sube tu propio CSV (interfaz) ---
uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Subir CSV")
salida_lote = widgets.Output()
display(uploader, salida_lote)

def al_subir(cambio):
    with salida_lote:
        salida_lote.clear_output()
        nombre, contenido = next(iter(uploader.value.items()))
        import io
        df = pd.read_csv(io.BytesIO(contenido["content"]))
        faltan = [c for c in FEATURES if c not in df.columns]
        if faltan:
            print(f"Faltan columnas: {faltan}")
            return
        pb2, pr2 = predecir(df)
        out = df[FEATURES].copy()
        out["P(diabetes)"] = np.round(pb2, 4)
        out["Predicción"] = np.where(pr2 == 1, "DIABETES", "NO DIABETES")
        display(out)

uploader.observe(al_subir, names="value")

## 📊 5. Respaldo del modelo (métricas y evidencias)

In [ ]:
# --- 5. Tabla comparativa + figuras del informe ---
display(tabla.style.format("{:.3f}").background_gradient(cmap="YlGn"))
mejor = tabla["Accuracy"].idxmax()
print(f"Mejor modelo: {mejor} (Accuracy {tabla.loc[mejor, 'Accuracy']:.3f})")
for img in ["m6_confusion.png", "m7_roc.png", "m8_accuracy.png"]:
    from IPython.display import Image
    display(Image(filename=str(BASE / "figuras" / img)))